# 02 LightGBM — entrenamiento por nivel (todos los niveles, incl. diarios de alta cardinalidad)

Versión enfocada de `02_model.ipynb`: mismo flujo (split, comparación base, selección de
features, SHAP, Optuna, modelo final), pero:

- **Solo LightGBM** entre los modelos de ML (se descartan XGBoost/CatBoost/HistGB/Ridge:
  no superaban a LightGBM en la comparación de `02_model.ipynb`, no vale la pena pagar su
  costo de RAM/tiempo acá).
- **Se mantienen los naive** (Naive, seasonal naive, drift, historical mean, moving
  average): son casi gratis y dan piso de referencia.
- **Se descartan los modelos estadísticos clásicos** (SARIMA/ETS/Theta/TBATS/Prophet):
  ajustan serie por serie, no escalan a niveles de miles de series (10/11/12).
- **LightGBM entrena con la API nativa (`lgb.Dataset`/`lgb.train`)**, no el wrapper
  sklearn: el dataset se binea una sola vez y se reutiliza en Optuna (hasta 1000 trials)
  y en el modelo final, en vez de re-binear el DataFrame en cada fit. El `X_train` en
  pandas se libera apenas se construye el `Dataset` bineado (~4x más chico, 1 byte/valor
  vs. 4 de `float32`). Este es el notebook pensado para poder correr niveles de alta
  cardinalidad (10/11/12) a grano diario sin quedarse sin RAM.

`02_model.ipynb` sigue existiendo para comparar familias de modelos completas en los
niveles agregados (1-9), donde el volumen de datos lo permite.

In [ ]:
import gc
import time

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from loguru import logger

pd.set_option("display.max_columns", None)

In [ ]:
!ls ../data/datasets/

# Data

In [ ]:
LEVEL = 'level_12_weekly_item_store'
OBJECTIVE = "tweedie"

GRAIN = 'weekly' if '_weekly_' in LEVEL else 'daily'
SEASON_LENGTH = {"daily": 7, "weekly": 52}[GRAIN]
RANDOM_STATE = 42

In [ ]:
df = pd.read_parquet(f'../data/datasets/dataset_{LEVEL}.parquet')

df = df.sort_values("date").copy()

df.head()

In [ ]:
df.info()

# Variables

In [ ]:
TARGET = "sales"

ID_COLS = ["series_id", "date"]
CUM_COLS = [c for c in df.columns if c.startswith("cum") and c[3:].isdigit()]
LEAKY_COLS = ["gross_sales"] + CUM_COLS

FEATURES = [c for c in df.columns if c not in ID_COLS + LEAKY_COLS + [TARGET]]

# Dims estáticas candidatas: no todos los niveles tienen todas (p.ej. nivel item
# no tiene store_id/state_id), así que se filtran a las presentes en el dataset.
CATEGORICAL_FEATURES = [
    c for c in [
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2",
    ] if c in df.columns
]

NUMERICAL_FEATURES = [c for c in FEATURES if c not in CATEGORICAL_FEATURES]

FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

print(len(FEATURES))

# Split

In [ ]:
from src.data import date_split

# daily: días. weekly: semanas (se convierte a días x7 antes de llamar a
# date_split, que siempre trabaja en días de calendario; así no queda ningún
# período cortado a la mitad).
TEST_PERIODS = 28 if GRAIN == "daily" else 4
VALID_PERIODS = 365 if GRAIN == "daily" else 52

TEST_DAYS = TEST_PERIODS if GRAIN == "daily" else TEST_PERIODS * 7
VALID_DAYS = VALID_PERIODS if GRAIN == "daily" else VALID_PERIODS * 7

data_split = date_split(df, valid_days=VALID_DAYS, test_days=TEST_DAYS)

train, valid, test = data_split.train, data_split.valid, data_split.test
first_date, last_date = data_split.first_date, data_split.last_date
valid_start, test_start = data_split.valid_start, data_split.test_start

data_split.log_summary()

In [ ]:
from src.data import build_feature_matrices

X_train, y_train, X_valid, y_valid, X_test, y_test = build_feature_matrices(
    df, data_split, FEATURES, CATEGORICAL_FEATURES, TARGET,
)

# train/valid/test solo se usan de acá en más para bookkeeping de evaluación (scales
# WRMSSE/MASE, pesos por precio, clip de cierres, gross_sales de reportes) -- no las
# ~100 columnas de features (esas ya están en X_train/X_valid/X_test). Sin este
# recorte, df + train/valid/test + X_train/X_valid/X_test conviven en RAM a la vez
# (~3x el dataset), lo que hace fallar por memoria a los niveles de mayor cardinalidad
# (10/11/12, sobre todo a grano diario). df/data_split ya no hacen falta: se liberan.
eval_cols = [c for c in ["series_id", "date", "sales", "gross_sales", "avg_sell_price", "is_store_closed"]
             if c in df.columns]
train = train[eval_cols].copy()
valid = valid[eval_cols].copy()
test = test[eval_cols].copy()
del df, data_split
gc.collect()

print(f"Cantidad features: {len(FEATURES)} ({len(CATEGORICAL_FEATURES)} categoricas)")

# Comparación de modelos (reducida)

Solo naive (piso de referencia, casi gratis) + LightGBM con hiperparámetros por
defecto. Se excluyen los modelos estadísticos serie-por-serie (SARIMA/ETS/Theta/
TBATS/Prophet, no escalan a miles de series) y el resto de ML (XGBoost/CatBoost/
HistGB/Ridge, ver `02_model.ipynb` para la comparación completa: no superaban a
LightGBM y no vale la pena pagar su costo de RAM/tiempo en niveles grandes).

In [ ]:
from src.evaluation import evaluate_predictions

model_results = []
predictions_valid = {}

def evaluate_model(name, y_pred_valid, fit_time=None, category=None):
    result = evaluate_predictions(train, valid, y_valid, y_pred_valid, name, fit_time=fit_time, category=category)
    model_results.append(result)
    predictions_valid[name] = y_pred_valid
    return result

## Baseline models

In [ ]:
from src.modeling import seasonal_naive, drift, historical_mean, moving_average

In [ ]:
t0 = time.perf_counter()
y_pred_naive = naive_last_value(train, valid, TARGET)
fit_time = time.perf_counter() - t0

evaluate_model("Naive (last value)", y_pred_naive, fit_time=fit_time, category="Naive")

In [ ]:
t0 = time.perf_counter()
y_pred_seasonal_short = seasonal_naive(train, valid, TARGET, season_length=SEASON_LENGTH)
fit_time = time.perf_counter() - t0
evaluate_model(f"Seasonal naive ({SEASON_LENGTH}{GRAIN[0]})", y_pred_seasonal_short, fit_time=fit_time, category="Naive")

if GRAIN == "daily":
    t0 = time.perf_counter()
    y_pred_seasonal_year = seasonal_naive(train, valid, TARGET, season_length=365)
    fit_time = time.perf_counter() - t0
    evaluate_model("Seasonal naive (365d)", y_pred_seasonal_year, fit_time=fit_time, category="Naive")

### Drift

In [ ]:
t0 = time.perf_counter()
y_pred_drift = drift(train, valid, TARGET)
fit_time = time.perf_counter() - t0

evaluate_model("Drift", y_pred_drift, fit_time=fit_time, category="Naive")

### Historical mean

In [ ]:
t0 = time.perf_counter()
y_pred_mean = historical_mean(train, valid, TARGET)
fit_time = time.perf_counter() - t0

evaluate_model("Historical mean", y_pred_mean, fit_time=fit_time, category="Naive")

### Moving average

In [ ]:
MA_WINDOWS = {"daily": [7, 14, 21, 28, 35], "weekly": [2, 3, 4, 6, 8,]}[GRAIN]

for window in MA_WINDOWS:
    t0 = time.perf_counter()
    y_pred_ma = moving_average(train, valid, TARGET, window=window)
    fit_time = time.perf_counter() - t0
    evaluate_model(f"Moving average ({window}{GRAIN[0]})", y_pred_ma, fit_time=fit_time, category="Naive")

## LightGBM (hiperparámetros por defecto)

In [ ]:
from src.modeling import fit_lightgbm

t0 = time.perf_counter()
lgbm_model = fit_lightgbm(X_train, y_train, CATEGORICAL_FEATURES, random_state=RANDOM_STATE)
fit_time = time.perf_counter() - t0

evaluate_model("LightGBM", lgbm_model.predict(X_valid), fit_time=fit_time, category="ML")

# Resultados

In [ ]:
def plot_barh_by_model(results_df, x, y="model", hue=None, title="", figsize=(8, 6)):
    """Barplot horizontal genérico para comparar modelos por una métrica."""
    df_sorted = results_df.sort_values(x, ascending=True)
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(df_sorted, x=x, y=y, hue=hue, order=df_sorted[y], ax=ax)
    ax.set_title(title, loc="left", pad=20)
    ax.set_xlabel("")
    ax.set_ylabel("")
    sns.despine()
    if hue:
        ax.legend(title=hue, loc="upper right")
    plt.tight_layout()
    plt.show()

In [ ]:
results_df = pd.DataFrame(model_results).sort_values("wrmsse")
results_df

### Métricas por serie para todos los modelos

`build_series_metrics` da WAPE/bias/RMSLE/tracking signal/WRMSSE/MASE/SPEC por serie para un modelo. `build_all_series_metrics` lo corre para todos los modelos de `predictions_valid` (mismo set de validación) y los concatena en un único DataFrame con columna `model`, para comparar múltiples métricas a nivel de serie entre modelos (no solo el agregado de `results_df`).

In [ ]:
from src.evaluation import build_all_series_metrics

series_metrics_df = build_all_series_metrics(train, valid, y_valid, predictions_valid, target_col=TARGET)
series_metrics_df.sort_values(["model", "gross_sales"], ascending=[True, False]).head(20)

In [ ]:
for col in ['wape', 'wrmsse', 'mae', 'rmse', 'mape', 'smape', 'bias', 'rmsle', 'tracking_signal', 'spec', 'mase', 'fit_time']:
    plot_barh_by_model(results_df, hue="category", x=col, y="model", title=f"{col} por modelo")

# Selección de features

El resto del notebook usa LightGBM (velocidad, soporte nativo de categóricas, compatible con SHAP), así que la selección se hace sobre `lgbm_model`. Se comparan dos criterios de importancia (gain del árbol vs. permutation importance sobre validación) y luego se aplica una eliminación hacia atrás (backward elimination): se van descartando las features menos importantes mientras el WRMSSE de validación no empeore más allá de una tolerancia.

## Importancia por ganancia (gain)

In [ ]:
importance_gain = (
    pd.Series(lgbm_model.booster_.feature_importance(importance_type="gain"), index=FEATURES)
    .sort_values(ascending=False)
)
importance_gain.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
importance_gain.head(30).sort_values().plot.barh(ax=ax)
ax.set_title("Importancia por ganancia (LightGBM, top 30)", loc="left")
sns.despine()
plt.tight_layout()
plt.show()

## Permutation importance

El gain puede sobreestimar features de alta cardinalidad o muy usadas para splits sin aportar demasiado al error final. La permutation importance mide directamente cuánto empeora el WAPE en validación al mezclar (shuffle) cada columna, así que es un criterio más fiel al desempeño real del modelo.

In [ ]:
from src.modeling import compute_permutation_importance

importance_perm = compute_permutation_importance(
    lgbm_model, X_valid, y_valid, FEATURES,
    sample_size=15_000, n_repeats=3, random_state=RANDOM_STATE,
)
importance_perm.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
importance_perm.head(30).sort_values().plot.barh(ax=ax)
ax.set_title("Permutation importance (RMSE, top 30)", loc="left")
sns.despine()
plt.tight_layout()
plt.show()

## Eliminación hacia atrás (backward elimination)

Se parte del ranking de permutation importance (de menor a mayor) y se intenta eliminar cada feature: si reentrenar sin ella no empeora el WRMSSE de validación más allá de `TOLERANCE`, se descarta definitivamente.

Nota: esta fase sigue usando el wrapper sklearn (`LGBMRegressor`) internamente -- cada intento prueba un subconjunto de columnas distinto, así que no hay un único `Dataset` bineado para reutilizar entre iteraciones (a diferencia de Optuna, donde las columnas no cambian). No se migró: no había ganancia real de memoria, cada fit ya se libera solo al terminar la iteración.

In [ ]:
from src.modeling import backward_feature_selection

selected_features, final_wrmsse, log_df = backward_feature_selection(
    X_train, y_train, X_valid, train, valid,
    FEATURES, CATEGORICAL_FEATURES, importance_perm,
    tolerance=0.001, random_state=RANDOM_STATE,
)

In [ ]:
baseline_wrmsse = log_df.loc[log_df["removed"].isna(), "wrmsse"].iloc[0]

# Trayectoria real: baseline + solo las eliminaciones aceptadas, en orden
trajectory = log_df[log_df["accepted"]].sort_values("n_features", ascending=False)
baseline_wrmsse = trajectory["wrmsse"].iloc[0]

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(trajectory, x="n_features", y="wrmsse", marker="o", ax=ax)
ax.invert_xaxis()
ax.axhline(baseline_wrmsse, color="grey", linestyle="--", linewidth=1, label="baseline (todas las features)")
ax.set_title("WRMSSE de validación durante la eliminación hacia atrás", loc="left")
ax.set_xlabel("Cantidad de features")
ax.set_ylabel("WRMSSE")
ax.legend(loc="upper right")
sns.despine()
plt.tight_layout()
plt.show()

## Features seleccionadas

Se fija `FEATURES` (y las variables derivadas `X_train`/`X_valid`/`X_test`/`CATEGORICAL_FEATURES`/`NUMERICAL_FEATURES`) al subconjunto seleccionado, para que el resto del notebook (Dataset nativo, SHAP, tuning con Optuna, modelo final) entrene sobre las features filtradas.

In [ ]:
print(f"Features descartadas ({len(FEATURES) - len(selected_features)}): "
      f"{sorted(set(FEATURES) - set(selected_features))}")

FEATURES = selected_features
CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c in FEATURES]
NUMERICAL_FEATURES = [c for c in NUMERICAL_FEATURES if c in FEATURES]

X_train = X_train[FEATURES]
X_valid = X_valid[FEATURES]
X_test = X_test[FEATURES]

print(f"Features finales: {len(FEATURES)} ({len(CATEGORICAL_FEATURES)} categóricas)")

# Modelo LightGBM (Dataset nativo)

De acá en más se arma el `lgb.Dataset` **una sola vez** (ya bineado) y se reutiliza tal
cual en el modelo simple, Optuna (hasta 1000 trials) y el modelo final -- en vez de que
cada `.fit()` vuelva a binear el mismo `X_train` desde pandas (lo que hacía el wrapper
sklearn en `02_model.ipynb`, 1000+ veces solo en Optuna). Apenas se construye `dtrain`,
se libera `X_train` de pandas: ya no hace falta, entrenar usa el `Dataset` bineado
(~1 byte/valor vs. 4 de `float32`, ~4x más chico). `X_valid` sí se mantiene -- a
diferencia del entrenamiento, `booster.predict()` necesita los datos crudos, no el
`Dataset`.

In [ ]:
import lightgbm as lgb

from src.evaluation import make_wrmsse_feval, make_wrmsse_metric

wrmsse_metric = make_wrmsse_metric(train, valid)
wrmsse_feval = make_wrmsse_feval(train, valid)

dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=CATEGORICAL_FEATURES, free_raw_data=True)
dvalid = lgb.Dataset(X_valid, label=y_valid, reference=dtrain, categorical_feature=CATEGORICAL_FEATURES, free_raw_data=True)
dtrain.construct()
dvalid.construct()

del X_train
gc.collect()

## Modelo simple

In [ ]:
t0 = time.perf_counter()
model = lgb.train(
    {"objective": "rmse", "metric": "None", "verbosity": -1, "seed": RANDOM_STATE},
    dtrain,
    num_boost_round=1_500,
    valid_sets=[dvalid],
    feval=wrmsse_feval,
    callbacks=[
        lgb.early_stopping(100, first_metric_only=True),
        lgb.log_evaluation(10),
    ],
)
fit_time = time.perf_counter() - t0

evaluate_model("LightGBM (selected features)", model.predict(X_valid), fit_time=fit_time, category="ML")

## Explicación modelo

In [ ]:
import shap

explainer = shap.TreeExplainer(model)

X_shap = X_test.sample(n=min(2000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_shap)

In [ ]:
shap_df = pd.DataFrame(shap_values, columns=X_shap.columns, index=X_shap.index)

importance_df = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

print(importance_df)

### Importancia global de features

In [ ]:
shap.summary_plot(shap_values, X_shap, plot_type="bar")

In [ ]:
shap.summary_plot(shap_values, X_shap)

### Dependencia por feature

In [ ]:
top_feature = X_shap.columns[np.argsort(-np.abs(shap_values).mean(axis=0))[0]]
print(f"Feature más importante: {top_feature}")

shap.dependence_plot(top_feature, shap_values, X_shap, interaction_index=None)

# Predicciones

In [ ]:
from src.evaluation import explain_prediction

In [ ]:
from src.evaluation import plot_forecast

In [ ]:
from src.evaluation import analizar_prediccion as _analizar_prediccion

def analizar_prediccion(series_id, date=None, max_display=10):
    return _analizar_prediccion(
        test, X_test, df_pred, TARGET, series_id,
        date=date, max_display=max_display, explainer=explainer,
    )

In [ ]:
from src.evaluation import build_predictions_report

metrics_test, df_pred = build_predictions_report(train, test, y_test, model.predict(X_test), target_col=TARGET)

print(f"Test WAPE: {metrics_test['wape']:.2%}")
print(f"Test WRMSSE: {metrics_test['wrmsse']:.4f}")

# Modelo optimizado

In [ ]:
import optuna
from optuna.integration import LightGBMPruningCallback

In [ ]:
def objective(trial):
    eval_metric = "tweedie" if OBJECTIVE == "tweedie" else "rmse"
    params = dict(
        objective=OBJECTIVE,
        metric=eval_metric,
        learning_rate=trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        num_leaves=trial.suggest_int("num_leaves", 31, 255),
        max_depth=trial.suggest_int("max_depth", 5, 10),
        min_child_samples=trial.suggest_int("min_child_samples", 20, 200),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 5, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 5, log=True),
        seed=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )

    if OBJECTIVE == "tweedie":
        params["tweedie_variance_power"] = trial.suggest_float(
            "tweedie_variance_power", 1.1, 1.9
        )

    # dtrain/dvalid ya están bineados (celda "Modelo LightGBM (Dataset nativo)"):
    # cada trial reentrena sobre el mismo Dataset, sin volver a pasar por pandas.
    booster = lgb.train(
        params, dtrain,
        num_boost_round=1_500,
        valid_sets=[dvalid],
        callbacks=[
            lgb.early_stopping(30, first_metric_only=True, verbose=False),
            LightGBMPruningCallback(trial, eval_metric),
        ],
    )

    y_pred = booster.predict(X_valid, num_iteration=booster.best_iteration)
    _, final_wrmsse, _ = wrmsse_metric(y_valid, y_pred)
    return final_wrmsse

In [ ]:
study = optuna.create_study(
    study_name=f"study_{LEVEL}",
    direction="minimize",
    storage="sqlite:///../artifacts/optuna_study.db",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(
        seed=42,
    ),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10, n_startup_trials=5),
)

study.optimize(
    objective,
    n_trials=1_000,
    timeout=60 * 15,
    show_progress_bar=True,
)

best_trial = study.best_trial
print(f"Mejor WRMSSE: {best_trial.value:.4f}")
print("Mejores hiperparámetros:", best_trial.params)

# Modelo final

In [ ]:
model_params = best_trial.params
eval_metric = "tweedie" if OBJECTIVE == "tweedie" else "rmse"

final_params = dict(
    objective=OBJECTIVE,
    metric=eval_metric,
    seed=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
    subsample_freq=1,
    **model_params,
)

t0 = time.perf_counter()
final_model = lgb.train(
    final_params, dtrain,
    num_boost_round=1_500,
    valid_sets=[dvalid],
    callbacks=[
        lgb.early_stopping(50, first_metric_only=True),
        lgb.log_evaluation(10),
    ],
)
fit_time = time.perf_counter() - t0

evaluate_model("LightGBM (selected features | Optuna)", final_model.predict(X_valid), fit_time=fit_time, category="ML")

## Resultados finales

In [ ]:
results_df = pd.DataFrame(model_results).sort_values("wrmsse").query('wape <= 0.15')
results_df

In [ ]:
for col in ['wape', 'wrmsse', 'mae', 'rmse', 'mape', 'smape', 'bias', 'rmsle', 'tracking_signal', 'spec', 'mase', 'fit_time']:
    plot_barh_by_model(results_df, hue="category", x=col, y="model", title=f"{col} por modelo")

# Exploración modelo final

In [ ]:
metrics_test_final, df_pred = build_predictions_report(train, test, y_test, final_model.predict(X_test), target_col=TARGET)

print(f"Test WAPE: {metrics_test_final['wape']:.2%}")
print(f"Test WRMSSE: {metrics_test_final['wrmsse']:.4f}")

# Exportar

In [ ]:
import joblib

feature_importance = (
    pd.Series(final_model.feature_importance(importance_type="gain"), index=FEATURES)
    .sort_values(ascending=False)
)

# Igual que en 02_model.ipynb: no se guardan df/X_train/y_train/X_valid/y_valid/valid
# (reconstruibles desde el parquet en data/datasets/ + este notebook, y
# notebooks/03_predictions.ipynb no los usa) -- guardarlos solo triplicaba el tamaño
# del artifact (y el pico de RAM al armarlo) sin necesidad.
# Sufijo "_lightgbm": para no pisar el artifact de 02_model.ipynb en el mismo nivel.
artifact = {
    "level": LEVEL,
    "objective": OBJECTIVE,
    "model": final_model,
    "model_params": model_params,
    "metrics_results": results_df,
    "X_test": X_test,
    "y_test": y_test,
    "train": train,
    "test": test,
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "numerical_features": NUMERICAL_FEATURES,
    "id_cols": ID_COLS,
    "leaky_cols": LEAKY_COLS,
    "target": TARGET,
    "feature_importance": feature_importance,
    "model_results": model_results,
    "results_df": results_df,
    "wape_valid": model_results[-1]["wape"],
    "wrmsse_valid": model_results[-1]["wrmsse"],
    "wape_test": metrics_test_final["wape"],
    "wrmsse_test": metrics_test_final["wrmsse"],
    "train_start": str(first_date.date()),
    "valid_start": str(valid_start.date()),
    "test_start": str(test_start.date()),
}

artifact_path = f"../artifacts/models/{LEVEL}_{TARGET}_lightgbm_artifact.pkl"
joblib.dump(artifact, artifact_path)
print(f"Artifact guardado en {artifact_path}")